# Interactive Brokers — Historical Data Pull

Connect to a locally running TWS (Trader Workstation) and pull historical bar data.
Saves to CSV format compatible with NautilusTrader's `BarDataWrangler`.

## Prerequisites
- TWS installed and running with API enabled (see setup instructions below)
- IB account with market data subscriptions

## TWS Installation & Setup

1. **Download TWS** from https://www.interactivebrokers.com/en/trading/download-tws.php
2. **Install** the `.dmg` file (Mac) and launch Trader Workstation
3. **Log in** with your IB username, password, and complete 2FA via the IBKR mobile app
4. **Enable API access:**
   - Go to **Edit > Global Configuration > API > Settings**
   - Check **Enable ActiveX and Socket Clients**
   - Set **Socket port** to `7496` (live) or `7497` (paper)
   - Uncheck **Read-Only API** if you want order execution (leave checked for data-only)
   - Click **Apply** and **OK**
5. **Keep TWS running** while using this notebook

In [ ]:
import datetime
import socket
from pathlib import Path

import pandas as pd

from nautilus_trader.adapters.interactive_brokers.common import IBContract
from nautilus_trader.adapters.interactive_brokers.historical.client import (
    HistoricInteractiveBrokersClient,
)

# === TWS Connection Settings ===
TWS_HOST = "127.0.0.1"
TWS_PORT = 7496  # 7496 = live, 7497 = paper
TWS_CLIENT_ID = 1

DATA_DIR = Path("../../../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)


def bars_to_dataframe(bars: list) -> pd.DataFrame:
    """Convert NautilusTrader Bar objects to a DataFrame with CSV-compatible columns."""
    return pd.DataFrame([
        {
            "timestamp": pd.Timestamp(bar.ts_event, unit="ns", tz="UTC"),
            "open": float(bar.open),
            "high": float(bar.high),
            "low": float(bar.low),
            "close": float(bar.close),
            "volume": float(bar.volume),
        }
        for bar in bars
    ])


def check_tws_connection(host: str, port: int) -> bool:
    """Check if TWS is accepting connections on the given host:port."""
    try:
        with socket.create_connection((host, port), timeout=3):
            return True
    except (ConnectionRefusedError, TimeoutError, OSError):
        return False


if check_tws_connection(TWS_HOST, TWS_PORT):
    print(f"TWS API detected at {TWS_HOST}:{TWS_PORT}")
else:
    print(f"WARNING: Cannot reach TWS at {TWS_HOST}:{TWS_PORT}")
    print()
    print("Please ensure:")
    print("  1. TWS (Trader Workstation) is running and you are logged in")
    print("  2. API is enabled: Edit > Global Configuration > API > Settings")
    print("     - 'Enable ActiveX and Socket Clients' is checked")
    print(f"     - Socket port is set to {TWS_PORT}")
    print("  3. If using paper trading, change TWS_PORT to 7497 above")
    print()
    print("Download TWS: https://www.interactivebrokers.com/en/trading/download-tws.php")

print(f"Data directory: {DATA_DIR.resolve()}")

## 1. Connect Historical Data Client

Connects to your locally running TWS for historical data requests.

> **Note:** If the connection fails, check the warning output from the cell above.

In [ ]:
client = HistoricInteractiveBrokersClient(
    host=TWS_HOST,
    port=TWS_PORT,
    client_id=TWS_CLIENT_ID,
)

await client.connect()
print(f"Connected to TWS at {TWS_HOST}:{TWS_PORT}")

## 2. Pull Historical Data

Configure the instrument, bar size, and date range below.
Two examples are provided: XAUUSD 1-minute and 5-minute bars.

### IB Rate Limits
- Max 60 requests per 10 minutes
- No identical requests within 15 seconds
- The client handles chunked requests automatically

### Available Bar Specifications
- `1-MINUTE-BID`, `1-MINUTE-LAST`, `1-MINUTE-MID`
- `5-MINUTE-BID`, `5-MINUTE-LAST`, `5-MINUTE-MID`
- `1-HOUR-BID`, `1-HOUR-LAST`, `1-HOUR-MID`
- `1-DAY-BID`, `1-DAY-LAST`, `1-DAY-MID`

In [ ]:
# XAUUSD (Spot Gold) — available for non-US residents
# For US residents, use: IBContract(secType="FUT", symbol="GC", exchange="COMEX", lastTradeDateOrContractMonth="YYYYMM")
xauusd_contract = IBContract(
    secType="CFD",
    symbol="XAUUSD",
    exchange="SMART",
    currency="USD",
)

# Date range — adjust as needed
# IB provides up to ~5 years of 1-minute data
end_date = datetime.datetime.now(tz=datetime.timezone.utc)
start_date = end_date - datetime.timedelta(days=365)  # 1 year

print(f"Contract: {xauusd_contract.symbol}")
print(f"Date range: {start_date.date()} to {end_date.date()}")

### Example 1: Pull 1-Minute Bars

In [ ]:
bars_1m = await client.request_bars(
    bar_specifications=["1-MINUTE-BID"],
    start_date_time=start_date,
    end_date_time=end_date,
    tz_name="UTC",
    contracts=[xauusd_contract],
    use_rth=False,  # Include extended hours for forex/CFDs
    timeout=120,
)

print(f"Received {len(bars_1m)} 1-minute bars")
if bars_1m:
    print(f"First bar: {bars_1m[0]}")
    print(f"Last bar:  {bars_1m[-1]}")

In [ ]:
if bars_1m:
    df_1m = bars_to_dataframe(bars_1m)

    filepath_1m = DATA_DIR / "xauusd_1m.csv"
    df_1m.to_csv(filepath_1m, index=False)
    print(f"Saved {len(df_1m)} bars to {filepath_1m.resolve()}")
    df_1m.head()

### Example 2: Pull 5-Minute Bars

In [ ]:
bars_5m = await client.request_bars(
    bar_specifications=["5-MINUTE-BID"],
    start_date_time=start_date,
    end_date_time=end_date,
    tz_name="UTC",
    contracts=[xauusd_contract],
    use_rth=False,
    timeout=120,
)

print(f"Received {len(bars_5m)} 5-minute bars")
if bars_5m:
    print(f"First bar: {bars_5m[0]}")
    print(f"Last bar:  {bars_5m[-1]}")

In [ ]:
if bars_5m:
    df_5m = bars_to_dataframe(bars_5m)

    filepath_5m = DATA_DIR / "xauusd_5m.csv"
    df_5m.to_csv(filepath_5m, index=False)
    print(f"Saved {len(df_5m)} bars to {filepath_5m.resolve()}")
    df_5m.head()

### Custom Pull

Modify the contract, bar spec, and date range below to pull any instrument.

In [ ]:
# === CONFIGURE YOUR PULL HERE ===
custom_contract = IBContract(
    secType="CFD",       # STK, FUT, OPT, CFD, CASH
    symbol="XAUUSD",     # IB symbol
    exchange="SMART",    # Exchange
    currency="USD",      # Currency
)

custom_bar_specs = ["1-MINUTE-BID"]  # Change timeframe here
custom_start = datetime.datetime.now(tz=datetime.timezone.utc) - datetime.timedelta(days=30)  # 30 days
custom_end = datetime.datetime.now(tz=datetime.timezone.utc)
custom_filename = "custom_data.csv"
# ================================

custom_bars = await client.request_bars(
    bar_specifications=custom_bar_specs,
    start_date_time=custom_start,
    end_date_time=custom_end,
    tz_name="UTC",
    contracts=[custom_contract],
    use_rth=False,
    timeout=120,
)

print(f"Received {len(custom_bars)} bars")

if custom_bars:
    df_custom = bars_to_dataframe(custom_bars)

    filepath_custom = DATA_DIR / custom_filename
    df_custom.to_csv(filepath_custom, index=False)
    print(f"Saved {len(df_custom)} bars to {filepath_custom.resolve()}")
    df_custom.head()